# Hybrid RAG
## a retrieval system with memory and output checks built from scratch

## What I'm building

In this notebook, I'm putting together a **small retrieval-augmented generation prototype** for research-style questions.

I'm not trying to make this production-ready. I'm deliberately keeping it small and understandable so I can experiment with the moving parts:

- fetching source material from a few URLs
- cleaning and chunking the text
- retrieving relevant chunks with both keyword and embedding search
- remembering what worked in earlier runs
- generating a structured answer
- checking whether the answer follows my evidence rules
- repairing the answer if it breaks those rules

The main point of this notebook is not scale. It is **clarity**.

I want a system I can inspect end-to-end:
- what I fetched
- what I chunked
- what I retrieved
- what the model cited
- what failed validation
- what got fixed

In [ ]:
!pip -q install openai openai-agents pydantic httpx beautifulsoup4 lxml scikit-learn numpy

In [ ]:
import os
import re
import json
import time
import getpass
import asyncio
import sqlite3
import hashlib
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import httpx
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from openai import AsyncOpenAI
from agents import Agent, Runner, SQLiteSession

## Why I'm starting here

Before I do anything "agentic", I need the boring foundation in place.

This notebook depends on three categories of tools:

- **web/content handling** so I can fetch pages and turn them into text
- **retrieval utilities** so I can index and search the chunks
- **agent orchestration** so I can split the workflow into planner / writer / fixer roles

I'm also using `pydantic` because I want typed outputs rather than free-form blobs.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required.")

client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ready.")

## My utility layer

I like to get small helpers out of the way early.

For this prototype, I need a few simple utilities:

- a stable hash function for IDs
- URL cleanup, because copied links are often messy
- HTML-to-text conversion
- chunking with overlap
- citation ID cleanup, because models often wrap IDs in punctuation

None of this is fancy. That is intentional.

In [ ]:
def make_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


def tidy_url(url: str) -> str:
    url = (url or "").strip()
    return url.rstrip(").,]\"'")


def html_to_plaintext(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def split_into_chunks(text: str, size: int = 1600, overlap: int = 320) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []

    if len(text) <= size:
        return [text]

    chunks = []
    step = max(1, size - overlap)

    for start in range(0, len(text), step):
        piece = text[start:start + size].strip()
        if piece:
            chunks.append(piece)
        if start + size >= len(text):
            break

    return chunks


def normalize_citation_id(value: str) -> str:
    value = (value or "").strip()
    return value.strip("[](){}<>\"'`")


def ensure_summary_has_citations(summary: str, citations: List[str], allowed_ids: List[str]) -> str:
    summary = (summary or "").strip()

    valid = []
    seen = set()

    for c in citations:
        c = normalize_citation_id(c)
        if c in allowed_ids and c not in seen:
            valid.append(c)
            seen.add(c)

    if len(valid) >= 2:
        return f"{summary} [{valid[0]}] [{valid[1]}]"

    fallback = allowed_ids[:2]
    if len(fallback) == 2:
        return f"{summary} [{fallback[0]}] [{fallback[1]}]"

    return summary


## My first design tradeoff

I'm chunking by **characters**, not tokens.

That is not ideal, but it's good enough for a notebook prototype as:

- it's easy to reason about
- it's easy to debug
- it avoids pulling in extra complexity too early

Later, if I want, I can replace this with token-aware or section-aware chunking.

For now I want to answer one question:
> can I get a working mini RAG loop with provenance discipline?

In [ ]:
async def fetch_pages(
    urls: List[str],
    timeout_seconds: float = 25.0,
    max_chars_per_page: int = 60000
) -> Dict[str, str]:
    headers = {"User-Agent": "MiniRAGPrototype/1.0"}
    cleaned_urls = [tidy_url(u) for u in urls]
    cleaned_urls = [u for u in cleaned_urls if u.startswith("http")]
    cleaned_urls = list(dict.fromkeys(cleaned_urls))

    results: Dict[str, str] = {}

    async with httpx.AsyncClient(
        timeout=timeout_seconds,
        follow_redirects=True,
        headers=headers
    ) as http:

        async def fetch_one(url: str):
            try:
                response = await http.get(url)
                response.raise_for_status()
                results[url] = html_to_plaintext(response.text)[:max_chars_per_page]
            except Exception as exc:
                results[url] = f"__FETCH_ERROR__ {type(exc).__name__}: {exc}"

        await asyncio.gather(*[fetch_one(url) for url in cleaned_urls])

    return results

## Why I'm deduplicating pages

When I work with small document sets, duplicate pages can quietly pollute retrieval.

For example:
- one URL might redirect to another
- two pages might contain near-identical content
- documentation pages may repeat the same sections

So I'm doing a crude dedupe pass based on a content fingerprint.

Not sophisticated. Just useful.

In [ ]:
def remove_duplicate_pages(pages: Dict[str, str]) -> Dict[str, str]:
    seen = set()
    unique_pages = {}

    for url, text in pages.items():
        if not isinstance(text, str) or text.startswith("__FETCH_ERROR__"):
            continue

        fingerprint = make_hash(text[:25000])
        if fingerprint in seen:
            continue

        seen.add(fingerprint)
        unique_pages[url] = text

    return unique_pages

## My data structures

I don't want this notebook to turn into a pile of anonymous dictionaries.

So I'm defining small typed models for:

- each chunk in my corpus
- each retrieval hit
- each group of hits per query

This makes the pipeline much easier to inspect and reason about.

In [ ]:
class Chunk(BaseModel):
    chunk_id: str
    source_url: str
    position: int
    text: str


class Hit(BaseModel):
    chunk_id: str
    source_url: str
    position: int
    keyword_score: float
    semantic_score: float
    combined_score: float
    text: str


class EvidenceGroup(BaseModel):
    query: str
    hits: List[Hit] = Field(default_factory=list)

## My memory layer

I want this prototype to remember a little bit about prior successful runs.

Not full chat history. Not full transcripts.

Just the useful parts:
- what question I answered
- what URLs I used
- what retrieval queries I generated
- which sources turned out useful

That is enough for a lightweight "episodic memory" effect.

In [ ]:
EPISODE_DB_PATH = "mini_rag_memory.db"


def init_episode_store():
    conn = sqlite3.connect(EPISODE_DB_PATH)
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS episodes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        created_at INTEGER NOT NULL,
        question TEXT NOT NULL,
        urls_json TEXT NOT NULL,
        queries_json TEXT NOT NULL,
        useful_sources_json TEXT NOT NULL
    )
    """)

    conn.commit()
    conn.close()


def save_episode(question: str, urls: List[str], queries: List[str], useful_sources: List[str]):
    conn = sqlite3.connect(EPISODE_DB_PATH)
    cur = conn.cursor()

    cur.execute(
        """
        INSERT INTO episodes(created_at, question, urls_json, queries_json, useful_sources_json)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            int(time.time()),
            question,
            json.dumps(urls),
            json.dumps(queries),
            json.dumps(useful_sources),
        ),
    )

    conn.commit()
    conn.close()


def recall_related_episodes(question: str, limit: int = 2) -> List[Dict[str, Any]]:
    conn = sqlite3.connect(EPISODE_DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        SELECT created_at, question, urls_json, queries_json, useful_sources_json
        FROM episodes
        ORDER BY created_at DESC
        LIMIT 200
    """)

    rows = cur.fetchall()
    conn.close()

    current_tokens = set(re.findall(r"[A-Za-z]{3,}", question.lower()))
    scored = []

    for created_at, old_question, urls_json, queries_json, useful_sources_json in rows:
        old_tokens = set(re.findall(r"[A-Za-z]{3,}", old_question.lower()))
        score = len(current_tokens & old_tokens) / max(1, len(current_tokens))

        scored.append({
            "score": score,
            "created_at": created_at,
            "question": old_question,
            "urls": json.loads(urls_json),
            "queries": json.loads(queries_json),
            "useful_sources": json.loads(useful_sources_json),
        })

    scored.sort(key=lambda item: item["score"], reverse=True)
    return [item for item in scored[:limit] if item["score"] > 0]


init_episode_store()
print("Episode store initialized.")

## Why I'm calling this "memory"

I'm using the word "memory" in a narrow sense here.

This notebook remembers **experience summaries**, not everything.

That gives me a simple pattern:
- answer a question
- keep a trace of what worked
- reuse it later when a similar question comes in

I like this because it keeps memory useful without making the prototype too heavy.

## My retrieval strategy

I don't want to rely on just one style of retrieval.

So I'm combining:

- **keyword-style retrieval** with TF-IDF
- **semantic retrieval** with embeddings

Then I merge the rankings.

This gives me a simple hybrid retriever without needing a dedicated vector database.

In [ ]:
class MiniHybridRetriever:
    def __init__(self):
        self.chunks: List[Chunk] = []
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.tfidf_matrix = None
        self.embedding_matrix: Optional[np.ndarray] = None

    def build_keyword_index(self):
        corpus = [chunk.text for chunk in self.chunks] if self.chunks else [""]
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=80000
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)

    def search_keywords(self, query: str, top_k: int) -> List[Tuple[int, float]]:
        if not self.chunks or self.vectorizer is None or self.tfidf_matrix is None:
            return []

        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = np.argsort(-scores)[:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    def set_embedding_matrix(self, matrix: np.ndarray):
        self.embedding_matrix = matrix.astype(np.float32)

    def search_semantic(self, query_embedding: np.ndarray, top_k: int) -> List[Tuple[int, float]]:
        if self.embedding_matrix is None or not self.chunks:
            return []

        docs = self.embedding_matrix
        query = query_embedding.astype(np.float32).reshape(1, -1)

        docs_norm = docs / (np.linalg.norm(docs, axis=1, keepdims=True) + 1e-12)
        query_norm = query / (np.linalg.norm(query, axis=1, keepdims=True) + 1e-12)

        scores = (docs_norm @ query_norm.T).flatten()
        top_indices = np.argsort(-scores)[:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

## Embeddings for dense retrieval

I'm using embeddings for the semantic side of retrieval.

This is what helps when the source wording and my query wording do not match exactly.

In other words:
- TF-IDF helps when the words line up
- embeddings help when the meaning lines up

That combination is good enough for a mini prototype.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"


async def embed_text_batch(texts: List[str]) -> np.ndarray:
    response = await client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype=np.float32)


async def embed_many_texts(
    texts: List[str],
    batch_size: int = 96,
    max_parallel: int = 3
) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1536), dtype=np.float32)

    texts = [(t or "")[:7000] for t in texts]
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    semaphore = asyncio.Semaphore(max_parallel)

    results = [None] * len(batches)

    async def embed_one_batch(batch_index: int, batch: List[str]):
        async with semaphore:
            results[batch_index] = await embed_text_batch(batch)

    await asyncio.gather(*[
        embed_one_batch(i, batch)
        for i, batch in enumerate(batches)
    ])

    matrices = [m for m in results if m is not None]
    output = np.vstack(matrices)

    if output.shape[0] != len(texts):
        raise RuntimeError(f"Embedding count mismatch: got {output.shape[0]}, expected {len(texts)}")

    return output


async def embed_one_query(query: str) -> np.ndarray:
    matrix = await embed_text_batch([query[:7000]])
    return matrix[0]

## How I'm merging keyword and semantic search

I'm not trying to calibrate keyword scores and semantic scores onto the same numeric scale.

That gets messy fast.

Instead, I'm using **rank fusion**:
- take the order from keyword retrieval
- take the order from semantic retrieval
- reward chunks that rank well in both

This is simple and robust enough for a notebook.

In [ ]:
def reciprocal_rank_fusion(rankings: List[List[int]], k: int = 60) -> Dict[int, float]:
    fused_scores: Dict[int, float] = {}

    for ranking in rankings:
        for rank_position, doc_index in enumerate(ranking):
            fused_scores[doc_index] = fused_scores.get(doc_index, 0.0) + 1.0 / (k + rank_position + 1)

    return fused_scores

In [ ]:
retriever = MiniHybridRetriever()
allowed_urls: List[str] = []


async def build_corpus_index(urls: List[str], max_chunks_per_url: int = 60):
    global retriever, allowed_urls

    raw_pages = await fetch_pages(urls)
    unique_pages = remove_duplicate_pages(raw_pages)

    chunks: List[Chunk] = []
    usable_urls = []

    for url, text in unique_pages.items():
        pieces = split_into_chunks(text)
        pieces = pieces[:max_chunks_per_url]

        for index, piece in enumerate(pieces):
            chunks.append(
                Chunk(
                    chunk_id=f"{make_hash(url)}:{index}",
                    source_url=url,
                    position=index,
                    text=piece
                )
            )

        if pieces:
            usable_urls.append(url)

    retriever = MiniHybridRetriever()
    retriever.chunks = chunks
    retriever.build_keyword_index()

    embedding_matrix = await embed_many_texts([chunk.text for chunk in chunks])
    retriever.set_embedding_matrix(embedding_matrix)

    allowed_urls = usable_urls
    print(f"Indexed {len(chunks)} chunks from {len(allowed_urls)} URLs.")

## What I consider "evidence"

For me, retrieval is not just "give me some chunks".

I want to preserve:
- which query pulled the chunk in
- where the chunk came from
- how it scored
- what the text actually was

That is why I'm grouping hits into evidence bundles per query.

In [ ]:
def make_evidence_group(
    query: str,
    keyword_hits: List[Tuple[int, float]],
    semantic_hits: List[Tuple[int, float]],
    per_query_limit: int = 10
) -> EvidenceGroup:
    keyword_order = [i for i, _ in keyword_hits]
    semantic_order = [i for i, _ in semantic_hits]

    keyword_scores = {i: s for i, s in keyword_hits}
    semantic_scores = {i: s for i, s in semantic_hits}

    fused = reciprocal_rank_fusion([keyword_order, semantic_order], k=60)
    top_indices = sorted(fused.keys(), key=lambda i: fused[i], reverse=True)[:per_query_limit]

    final_hits = []
    for idx in top_indices:
        chunk = retriever.chunks[idx]
        final_hits.append(
            Hit(
                chunk_id=chunk.chunk_id,
                source_url=chunk.source_url,
                position=chunk.position,
                keyword_score=float(keyword_scores.get(idx, 0.0)),
                semantic_score=float(semantic_scores.get(idx, 0.0)),
                combined_score=float(fused.get(idx, 0.0)),
                text=chunk.text
            )
        )

    return EvidenceGroup(query=query, hits=final_hits)

In [ ]:
async def collect_evidence(
    queries: List[str],
    per_query_limit: int = 10,
    keyword_pool: int = 60,
    semantic_pool: int = 60
):
    evidence_groups: List[EvidenceGroup] = []
    source_frequency: Dict[str, int] = {}
    cited_chunk_ids: List[str] = []

    for query in queries:
        keyword_hits = retriever.search_keywords(query, keyword_pool)
        query_embedding = await embed_one_query(query)
        semantic_hits = retriever.search_semantic(query_embedding, semantic_pool)

        group = make_evidence_group(
            query=query,
            keyword_hits=keyword_hits,
            semantic_hits=semantic_hits,
            per_query_limit=per_query_limit
        )
        evidence_groups.append(group)

        for hit in group.hits:
            source_frequency[hit.source_url] = source_frequency.get(hit.source_url, 0) + 1
            cited_chunk_ids.append(hit.chunk_id)

    useful_sources = sorted(source_frequency.keys(), key=lambda url: source_frequency[url], reverse=True)

    seen = set()
    allowed_chunk_ids = []
    for chunk_id in cited_chunk_ids:
        if chunk_id not in seen:
            allowed_chunk_ids.append(chunk_id)
            seen.add(chunk_id)

    return evidence_groups, useful_sources, allowed_chunk_ids

## My answer contract

I want the final output to be structured.

Not because structure is fashionable, but because I want predictable fields for:
- display
- validation
- debugging
- later reuse

So I'm defining one schema for planning and another for the final answer.

In [ ]:
class RetrievalPlan(BaseModel):
    objective: str
    subtasks: List[str]
    retrieval_queries: List[str]
    quality_checks: List[str]


class FinalAnswer(BaseModel):
    title: str
    executive_summary: str
    architecture: List[str]
    retrieval_strategy: List[str]
    workflow: List[str]
    implementation_notes: List[str]
    limitations: List[str]
    citations: List[str]
    sources: List[str]

## My validation philosophy

I'm not trying to prove every sentence is perfectly grounded.

What I do want is a simple standard:

- the answer should only cite chunks I actually retrieved
- the answer should only mention sources I actually allowed
- the answer should cite enough evidence to avoid being hand-wavy
- the summary should visibly show evidence, not just the body

This gives me a basic output-discipline layer.

In [ ]:
def clean_answer(answer: FinalAnswer, allowed_chunk_ids: List[str]) -> FinalAnswer:
    cleaned_citations = []
    seen = set()

    for c in answer.citations:
        c = normalize_citation_id(c)
        if c in allowed_chunk_ids and c not in seen:
            cleaned_citations.append(c)
            seen.add(c)

    answer.citations = cleaned_citations
    answer.sources = [url for url in answer.sources if url in allowed_urls]

    visible_summary_citations = re.findall(r"\[([^\]]+)\]", answer.executive_summary or "")
    if len(visible_summary_citations) < 2:
        answer.executive_summary = ensure_summary_has_citations(
            answer.executive_summary,
            answer.citations,
            allowed_chunk_ids
        )

    return answer


def validate_answer(answer: FinalAnswer, allowed_chunk_ids: List[str]):
    invalid_sources = [url for url in answer.sources if url not in allowed_urls]
    if invalid_sources:
        raise ValueError(f"Found disallowed sources: {invalid_sources}")

    invalid_citations = [c for c in answer.citations if c not in allowed_chunk_ids]
    if invalid_citations:
        raise ValueError(f"Found disallowed citations: {invalid_citations}")

    if len(set(answer.citations)) < 6:
        raise ValueError("Need at least 6 distinct citations in the final answer.")

    summary_refs = re.findall(r"\[([^\]]+)\]", answer.executive_summary or "")
    summary_refs = [normalize_citation_id(c) for c in summary_refs]
    summary_refs = [c for c in summary_refs if c in allowed_chunk_ids]

    if len(set(summary_refs)) < 2:
        raise ValueError("Executive summary must contain at least 2 valid citations.")

## My three-agent split

I'm using three agents because they each do a different kind of thinking:

### Planner
I use this agent to generate sub-queries and search angles.

### Writer
I use this agent to turn evidence into a structured answer.

### Fixer
I use this agent when the writer gives me an answer that breaks my rules.

This separation keeps the prompts cleaner and makes it easier to diagnose failures.

In [ ]:
planner_agent = Agent(
    name="retrieval_planner",
    model="gpt-4o-mini",
    output_type=RetrievalPlan,
    instructions="""
I am using you inside a small research-oriented RAG prototype.

Your task is to plan retrieval, not to write the final answer.

You should:
- understand the user's question
- break it into sub-problems
- generate 10 to 16 retrieval queries
- propose quality checks for the final answer

You must think in terms of recall and coverage.
Assume the system may only use the URLs I provide.
"""
)

writer_agent = Agent(
    name="structured_writer",
    model="gpt-4o-mini",
    output_type=FinalAnswer,
    instructions="""
You are the answer-writing step in my mini RAG prototype.

You will receive:
- the user question
- allowed source URLs
- allowed chunk IDs
- evidence groups containing retrieved text

Your answer must:
- stay within the evidence
- cite only allowed chunk IDs
- use only allowed URLs in the sources field
- include at least 6 unique citations
- include at least 2 visible citations in the executive summary
- be concrete and implementation-oriented
"""
)

repair_agent = Agent(
    name="answer_repair",
    model="gpt-4o-mini",
    output_type=FinalAnswer,
    instructions="""
You repair answers that failed my prototype validation checks.

You will receive:
- the invalid answer
- the validation error
- the allowed URLs
- the allowed chunk IDs

Your job is to fix the answer with minimal changes while making it pass validation.
"""
)

session = SQLiteSession("mini-rag-session")
print("Agents ready.")

## Why I'm also using session memory

I now have two kinds of memory in this notebook:

### 1. Session memory
This comes from the Agents SDK session and helps preserve conversational context across agent runs.

### 2. Episode memory
This is my own SQLite store of prior successful research episodes.

I like keeping both because they serve different purposes:
- session memory helps during the current workflow
- episode memory helps across similar future tasks

## The end-to-end workflow

This final orchestration cell is where I connect everything together.

The sequence is:

1. build the corpus
2. recall similar past episodes
3. ask the planner for sub-queries
4. retrieve evidence with hybrid search
5. ask the writer for a structured answer
6. validate the answer
7. if needed, repair it
8. store what worked

That is my full mini RAG loop.

In [ ]:
async def run_mini_rag(question: str, urls: List[str], max_repairs: int = 2) -> FinalAnswer:
    await build_corpus_index(urls)

    recalled = recall_related_episodes(question, limit=2)
    recalled_json = json.dumps(recalled, indent=2)[:2000]

    plan_result = await Runner.run(
        planner_agent,
        (
            f"Question:\n{question}\n\n"
            f"Allowed URLs:\n{json.dumps(allowed_urls, indent=2)}\n\n"
            f"Similar past episodes:\n{recalled_json}\n"
        ),
        session=session
    )

    plan: RetrievalPlan = plan_result.final_output
    retrieval_queries = (plan.retrieval_queries or [])[:16]

    if not retrieval_queries:
        raise RuntimeError("Planner did not produce any retrieval queries.")

    evidence_groups, useful_sources, allowed_chunk_ids = await collect_evidence(retrieval_queries)

    evidence_payload = json.dumps([group.model_dump() for group in evidence_groups], indent=2)[:16000]
    allowed_chunk_payload = json.dumps(allowed_chunk_ids[:200], indent=2)

    draft_result = await Runner.run(
        writer_agent,
        (
            f"Question:\n{question}\n\n"
            f"Allowed URLs:\n{json.dumps(allowed_urls, indent=2)}\n\n"
            f"Allowed chunk IDs:\n{allowed_chunk_payload}\n\n"
            f"Evidence groups:\n{evidence_payload}\n"
        ),
        session=session
    )

    answer: FinalAnswer = clean_answer(draft_result.final_output, allowed_chunk_ids)

    last_error = None

    for attempt in range(max_repairs + 1):
        try:
            validate_answer(answer, allowed_chunk_ids)

            save_episode(
                question=question,
                urls=allowed_urls,
                queries=retrieval_queries,
                useful_sources=useful_sources[:10]
            )

            print(f"Answer validated after {attempt} repair rounds.")
            return answer

        except Exception as exc:
            last_error = str(exc)
            print(f"Validation failed on attempt {attempt + 1}: {last_error}")

            if attempt >= max_repairs:
                break

            repair_result = await Runner.run(
                repair_agent,
                (
                    f"Validation error:\n{last_error}\n\n"
                    f"Allowed URLs:\n{json.dumps(allowed_urls, indent=2)}\n\n"
                    f"Allowed chunk IDs:\n{allowed_chunk_payload}\n\n"
                    f"Draft answer:\n{json.dumps(answer.model_dump(), indent=2)}\n"
                ),
                session=session
            )

            answer = clean_answer(repair_result.final_output, allowed_chunk_ids)

    raise RuntimeError(f"Could not produce a valid answer. Last error: {last_error}")


## A simple test run

Now I want to try the prototype on a small set of documentation pages.

I’m choosing a compact set of source URLs so I can inspect the behavior more easily.

In [ ]:
question = """
I want a compact architecture for a prototype RAG system that uses hybrid retrieval,
lightweight memory, and answer validation. Explain the architecture, retrieval approach,
workflow, and implementation notes.
"""

urls = [
    "https://openai.github.io/openai-agents-python/",
    "https://openai.github.io/openai-agents-python/agents/",
    "https://openai.github.io/openai-agents-python/running_agents/",
    "https://openai.github.io/openai-agents-python/sessions/",
]

result = await run_mini_rag(question, urls, max_repairs=2)

In [ ]:
print("\n=== TITLE ===")
print(result.title)

print("\n=== EXECUTIVE SUMMARY ===")
print(result.executive_summary)

print("\n=== ARCHITECTURE ===")
for item in result.architecture:
    print("-", item)

print("\n=== RETRIEVAL STRATEGY ===")
for item in result.retrieval_strategy:
    print("-", item)

print("\n=== WORKFLOW ===")
for item in result.workflow:
    print("-", item)

print("\n=== IMPLEMENTATION NOTES ===")
for item in result.implementation_notes:
    print("-", item)

print("\n=== LIMITATIONS ===")
for item in result.limitations:
    print("-", item)

print("\n=== CITATIONS ===")
print(result.citations)

print("\n=== SOURCES ===")
print(result.sources)


## What I learned from this prototype

This notebook gives me a compact but complete RAG loop:

- I fetch a small corpus
- I turn it into chunks
- I retrieve with both lexical and semantic search
- I remember which sources helped in earlier runs
- I generate a typed answer
- I validate that answer against my provenance rules
- I repair it when it fails

What I like most is that every stage is inspectable.

If something goes wrong, I can ask:
- did I fetch bad pages?
- did chunking lose context?
- did the planner generate weak queries?
- did retrieval pull the wrong chunks?
- did the answer break citation rules?
- did the repair step actually fix the issue?

That is exactly what I want from a prototype notebook.

## What this is not

I’m intentionally not treating this as a production system.

It does **not** yet handle:
- large corpora
- serious document parsing
- high-quality reranking
- full grounding verification
- cost-aware caching
- access control
- observability
- evaluation pipelines

That is okay.

The point of this notebook is to make the core loop real enough that I can iterate on it.

## If you plan to use this in Production

For use on production, the first things I would improve are:

1. better chunking  
2. stronger reranking  
3. more useful episode recall  
4. saving final validated answers into memory  
5. a richer citation format for human readability  

For now, though, this is enough to prove the core idea.

Before making any improvements though, I'm interested in making the planning layer to support
optimizing for 
- **Low Cost** - fewest web calls / fewer tokens
- **High Certainity** - more retrieval + verification